In [ ]:
import socket
from pathlib import Path
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from IPython.display import display

%matplotlib inline

import pickle

from vi_rnn.saving import load_model, CPU_Unpickler
from vi_rnn.data_utils import make_all_trials
from vi_rnn.fixed_points import run_scify, check_fixed_points_one_step
from fig_utils.transformed_rnn import (
    transformed_rnn,
    check_reduced_frozen_params,
    check_reduced_frozen_dynamics,
)

In [ ]:
from fig_utils.fixed_points import (
    get_multi_scale_jitter,
)
from fig_utils.plots import (
    plot_fixed_points_by_macaque,
    plot_fixed_points_subspaces,
    plot_frozen_latents,
)
from fig_utils.plots import plot_basis_2d_subspaces, position_slice_at_time

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# load the basis dataframe
df_basii = pickle.load(open("../data/processed/df_basii.pkl", "rb"))

In [ ]:
# --- controls ---

# --- task / basis ---
n_pos = 3
n_pcs_time = 2
n_stim = 6
t_decode = -1

# --- simulation ---
n_duplications = 1
noise_scale = 1
generate_plots = True

# --- freezing / fixed points ---
fixed_inds = np.arange(n_pcs_time)
freeze_time_steps = [50, 60, 70]  # time bin where clamping starts
freeze_noise = 0.1
n_ts_plot_frozen = 140

# --- optimizer init (scipy search) ---
noise_scales = [0.1, 0.5, 2.0, 4.0, 10, 20]
samples_per_scale = 500

# --- style ---
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

# --- run ---
run = False

In [ ]:
# get task_params of one of the models
task_params_file = str(out_dir) + "/" + model_dirs[0] + "_task_params.pkl"
with open(task_params_file, "rb") as f:
    task_params = CPU_Unpickler(f).load()
bin_size = task_params["bin_size"]

u, _, labels, delay_ends = make_all_trials(
    task_params,
    dur=3.55,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=-1,
    bin_size=0.05,
    interval_dur="mean",
    delay_dur="mean",
)
# also generate a longer sequence to get a better estimate of the fixed points
u2, _, labels2, delay_ends2 = make_all_trials(
    task_params,
    dur=7,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=-1,
    bin_size=0.05,
    interval_dur="mean",
    delay_dur="mean",
)
u2 = np.concatenate([u2] * n_duplications, axis=0)
labels2 = np.concatenate([labels2] * n_duplications, axis=0)

In [ ]:
if run:
    rows = []

    for model_dir in model_dirs[:10]:
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=True, backward_compat=False
        )

        row = df_basii.loc[df_basii["name"] == name]
        if row.empty:
            print(f"skip {name}: not in df_basii")
            continue
        print("model name: ", name)
        print("macaque: ", task_params["sessions"][0][5:10])

        # extract the linear transformation
        A_comb_np = row["A_comb_np"].values[0]
        b_comb_np = row["b_comb_np"].values[0]
        # create an RNN expression in the new basis
        rnn_orth = transformed_rnn(vae, A_comb_np, b_comb_np)

        # to check if the basis is good
        if generate_plots:
            zs = rnn_orth.simulate(u)
            z_by_pos = [
                position_slice_at_time(zs, t_decode, i, n_pcs_time)
                for i in range(n_pos)
            ]
            c_by_pos = [labels[:, i] for i in range(n_pos)]
            plot_basis_2d_subspaces(
                z_by_pos, c_by_pos, cmap=cmap, n_stim=n_stim, n_pos=n_pos
            )

        Z_fps_list = []
        N_fps_list = []
        max_eig_mags_list = []
        for freeze_time_step in freeze_time_steps:
            print("freeze_time_step: ", freeze_time_step)
            # simulate the full model
            Z = rnn_orth.simulate(
                u2,
                z0=None,
                noise_scale=noise_scale,
                freeze_indices=[],
                freeze_time_step=-1,
                freeze_noise=freeze_noise,
                freeze="mean",
            )
            # simulate the frozen model
            Z_frozen = rnn_orth.simulate(
                u2,
                z0=None,
                noise_scale=noise_scale,
                freeze_indices=[*fixed_inds],
                freeze_time_step=freeze_time_step,
                freeze_noise=freeze_noise,
                freeze="mean",
            )

            if generate_plots:
                plot_frozen_latents(
                    Z,
                    Z_frozen,
                    labels2,
                    cmap,
                    n_pos,
                    n_ts_plot_frozen,
                    freeze_bin=freeze_time_step,
                    bin_size=bin_size,
                    n_stim=n_stim,
                )

            # extract the value of the frozen latents
            z_fixed = Z_frozen[:, fixed_inds, freeze_time_step + 10 :].mean(axis=(0, 2))

            # get a model in dim_z - n_frozen_dims and find fixed points
            W1_m = rnn_orth.W1_m
            A_mm = rnn_orth.tau_m
            A_mf = rnn_orth.tau_mf
            W2_m = rnn_orth.W2_m
            W2_f = rnn_orth.W2_f
            h1_m = rnn_orth.h1_m
            h2 = rnn_orth.h2
            I = rnn_orth.pI
            v = np.array([0, 0, 1])

            h1_tilde = h1_m + A_mf @ z_fixed
            h2_tilde = h2 + W2_f @ z_fixed + I @ v
            W2_full = rnn_orth.W2
            x = Z_frozen[:, :, -1] @ W2_full.T + h2 + I.dot(v)
            D_init = np.array(x > 0).astype("uint8")
            moving_inds = np.setdiff1d(np.arange(vae.dim_z), fixed_inds)

            unique_patterns = get_multi_scale_jitter(
                Z_frozen[:, :, -1],
                W2_full=W2_full,
                h2=h2,
                I=I,
                v=v,
                scales=noise_scales,
                samples_per_scale=samples_per_scale,
            )
            D_combined = np.vstack([D_init, unique_patterns])
            print(f"Total unique patterns for scify: {len(D_combined)}")

            check_reduced_frozen_params(
                rnn_orth,
                fixed_inds,
                z_fixed,
                v,
                A_mm,
                W1_m,
                W2_m,
                h1_tilde,
                h2_tilde,
            )

            found_lower_orders, found_eigvals, n_inverses = run_scify(
                A=A_mm,
                W1=W1_m,
                W2=W2_m,
                h1=h1_tilde,
                h2=h2_tilde,
                order=1,
                inner_loop_iterations=10,
                round_dec=6,
                n_inverses_max=100000,
                initial_states=D_combined,
            )
            check_fixed_points_one_step(
                A_mm, W1_m, W2_m, h1_tilde, h2_tilde, found_lower_orders[0]
            )
            check_reduced_frozen_dynamics(
                rnn_orth, fixed_inds, z_fixed, v, np.array(found_lower_orders[0])[:, 0]
            )

            evs = np.array(found_eigvals[0])
            Z_fps = np.array(found_lower_orders[0])[:, 0]
            max_eig_mags = [np.max(np.abs(e)) for e in evs]
            # first flush
            print("\n", flush=True)

            print(
                f"Amount of stable fixed points: {np.sum(np.array(max_eig_mags) < 1)}"
            )
            print(
                f"Amount of unstable fixed points: {np.sum(np.array(max_eig_mags) > 1)}"
            )

            if generate_plots:
                if zs is None:
                    zs = rnn_orth.simulate(u)
                plot_fixed_points_subspaces(
                    Z_fps,
                    max_eig_mags,
                    zs,
                    labels,
                    t_decode=t_decode,
                    n_pcs_time=n_pcs_time,
                    cmap=cmap,
                    n_stim=n_stim,
                    n_pos=n_pos,
                )
            Z_fps_list.append(Z_fps)
            N_fps_list.append(len(Z_fps))
            max_eig_mags_list.append(max_eig_mags)
        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "macaque": task_params["sessions"][0][5:10],
                "fixed_points": Z_fps_list,
                "max_eig_mags": max_eig_mags_list,
                "n_fixed_points": N_fps_list,
                "freeze_time_steps": freeze_time_steps,
                "z_fixed": z_fixed,
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    df.to_pickle("../data/processed/df_fixed_points.pkl")
else:
    df = pd.read_pickle("../data/processed/df_fixed_points.pkl")

In [ ]:
plot_fixed_points_by_macaque(
    df,
    bin_size=bin_size,
    box_w=0.8,
    save_path="../paper_figures/fixed_points_by_macaque.pdf",
)

In [ ]:
i = list(df.iloc[0]["freeze_time_steps"]).index(
    60
)  # same index for all rows if steps are shared
idx = df["n_fixed_points"].apply(lambda xs: xs[i]).idxmax()
row = df.loc[idx]
print(row["name"], row["n_fixed_points"][i], "at freeze_time_step=60")